In [ ]:
# Import Packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

%matplotlib inline

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.datasets import load_iris

iris_data = load_iris()
iris_data.keys()
X = iris_data['data']
y = iris_data['target']

### Zadanie 1

Przygotować podzbiory treningowy (60% danych) i testowych (40% danych). Wytrenować model [k najbliższych sasiadów](https://pl.wikipedia.org/wiki/K_najbli%C5%BCszych_s%C4%85siad%C3%B3w) (lub algorytm k-nn z ang. *k nearest neighbours*) dla k = 3. Policzyć za pomocą wytrenowanego modelu predykcje dla danych testowych. Obliczyć dokładność (ang. *accuracy*) modelu na danych testowych. Policzyć i przedstawić na wykresie macierz pomyłek.

Sprawdzić wyniki przy i bez użycia stratyfikacji.

In [ ]:
# Miejsce na rozwiązanie
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

# podział danych
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.4)

# trening modelu
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

# accuracy score 
accuracy = accuracy_score(y_test, model.predict(X_test))
print(accuracy)
# macierz pomyłek 
results = confusion_matrix(y_test, model.predict(X_test))

disp = ConfusionMatrixDisplay(results)
disp.plot()



### Zadanie 2

Przprowadzić 5-krotną walidację krzyżową dla modelu k najbliższych sąsiadów celem oceny trafności prognostycznej modelu predykcyjnego. Policzyć dokładność modelu dla poszczególnych walidacji i dokładność średnią.

Podpowiedź: Funkcja [sklearn.model_selection.ross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, balanced_accuracy_score
# Miejsce na rozwiązanie

results = cross_val_score(
    estimator=KNeighborsClassifier(),
    X = X,
    y = y,
    cv = 5,
    # scores = accuracy_score
    #TODO - jak ustawić te score funkcje 
).reshape(-1, 1)

# pytanie o co chodzi tutaja

# dokładność średnia 
print(results, results.mean(), results.std()) 

### Zadanie 3

Poszukamy optymalnego ustawienia wartości hiperparametru k. Do tej pory wartością tego hiperparametru było 3 i nie było żadnego uzasadnienia dla takiego wyboru. Proszę znaleźć optymalną wartość k wykorzystując funkcję [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

In [ ]:
# Miejsce na rozwiązanie
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score

clf = GridSearchCV(KNeighborsClassifier(), {'n_neighbors': [i for i in range(1, 40)]}, scoring=['accuracy', 'balanced_accuracy'], refit='accuracy')
clf.fit(X, y) # dałem wszyskie
# clf # metodami można to wydobyć później
pd.DataFrame(clf.cv_results_)


### Zadanie 4

Przygotować podzbiory treningowy (80% danych) i testowych (20% danych). Wytrenować modele oparte o Maszynę Wektorów Nośnych(ang. *Support Vector Machine*) używając klasy [sklearn.svm.SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) z różnymi ustawieniami parametru "kernel". Policzyć za pomocą wytrenowanych modeli predykcje dla danych testowych. Obliczyć dokładność (ang. *accuracy*) modelu na danych testowych. Zapisać wytrenowany model do pliku, następnie wczytać go z pliku i ponownie policzyć predykcje dla danych [[3, 2, 1, 0.2], [  4.9, 2.2, 3.8, 1.1 ], [  5.3, 2.5, 4.6, 1.9 ]].

Support Vector Machine classifier

In [ ]:
# Miejsce na rozwiązanie
from pathlib import Path
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
# zapisywanie tych plików 
import pickle

# ścieżka 
folder_path = Path('trained_models')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2) 

kernels = {'Linear kernel': 'linear', 'Polynomial Kernel': 'poly', 'Radial basis kernel':'rbf', 'Sigmoid Kernel': 'sigmoid'}

# ocena modeli - można było wykorzystać Grid, ale tutaj łatwiej mi operować już wytrenowanymi modelami

# wytrenowanie modeli 
trained_models = dict() # it will contain all trained models 
for name, kernel in kernels.items():
    model = SVC(kernel=kernel)
    model.fit(X_train, y_train)
    score = accuracy_score(y_test, model.predict(X_test))
    file_path = Path(folder_path, name)
    pickle.dump(model, open(file_path, 'wb'))
    trained_models[name] = [model, score]

# część z zapisem i wyliczeniem predykcji
data = [[3, 2, 1, 0.2], [  4.9, 2.2, 3.8, 1.1 ], [  5.3, 2.5, 4.6, 1.9 ]]
# importing files 
for name in trained_models.keys():
    model_path = Path(folder_path, name)
    model_loaded = pickle.load(open(model_path, 'rb'))
    prediction = model_loaded.predict(data)
    trained_models[name].append(prediction)


pd.DataFrame(trained_models).T.rename(columns={0: 'Model', 1: 'accuracy', 2: 'prediction on new data'})



In [ ]:
!mkdir trained_models